# Przetwarzanie i dzielenie plików PDF
Ten notebook rozwija przepływ pracy z analizy dokumentów z `09.ReadDocument/03.pdf-reader.ipynb` i rozszerza go o strategie dzielenia tekstu, które można wykorzystać w dalszych potokach wyszukiwania.
1. **Narzędzia ekstrakcji** – lokalne (`PyPDF2`) i zdalne (Azure Document Intelligence) pomocniki do zamiany plików PDF na Markdown i zwykły tekst.
2. **Czyszczenie i inspekcja** – współdzielone funkcje do zliczania tokenów i szybkiego podglądu fragmentów.
3. **Przepisy na dzielenie tekstu** – przykłady dzielenia na podstawie nagłówków, rekurencyjnych znaków, Unstructured i Semchunk do eksperymentowania ze strategiami segmentacji.

In [ ]:
%pip install logging tiktoken azure-ai-documentintelligence azure-core azure-identity PyPDF2 python-dotenv langchain langchain-text-splitters "unstructured[md]" semchunk markdown

In [ ]:
FILE_NAME = "data/pdf/sample-pdf.pdf"

In [ ]:
import logging
import os
import re
from dataclasses import dataclass
from io import BytesIO
from typing import Iterable, Sequence

import tiktoken
from dotenv import load_dotenv
from PyPDF2 import PdfReader
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeResult
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import CharacterTextSplitter

from langchain_text_splitters import RecursiveCharacterTextSplitter
from unstructured.chunking.title import chunk_by_title
from unstructured.partition.md import partition_md

load_dotenv()
encoding = tiktoken.get_encoding("cl100k_base")


@dataclass(frozen=True)
class ChunkingConfig:
    markdown_headers: tuple[tuple[str, str], ...] = (
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
        ("####", "Header 4"),
        ("#####", "Header 5"),
        ("######", "Header 6"),
        ("#######", "Header 7"),
        ("########", "Header 8"),
    )
    chunk_size: int = 750
    chunk_overlap: int = 50
    recursive_separators: tuple[str, ...] = (".", "!", "\n\n", "\n", " ", "")
    semchunk_model: str = "gpt-4o-mini"


CONFIG = ChunkingConfig(
    semchunk_model=os.getenv("SEMCHUNK_MODEL", "gpt-4o-mini"),
)

if not logging.getLogger().handlers:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        handlers=[
            logging.FileHandler("debug.log"),
            logging.StreamHandler(),
        ],
    )

logger = logging.getLogger(__name__)


def build_markdown_header_splitter(config: ChunkingConfig) -> MarkdownHeaderTextSplitter:
    """Zwraca splitter nagłówków Markdown skonfigurowany zgodnie z podanymi domyślnymi ustawieniami."""
    return MarkdownHeaderTextSplitter(
        headers_to_split_on=config.markdown_headers,
        strip_headers=False,
    )


def build_recursive_splitter(config: ChunkingConfig) -> RecursiveCharacterTextSplitter:
    """Zwraca rekurencyjny splitter znaków do dodatkowych przebiegów dzielenia tekstu."""
    return RecursiveCharacterTextSplitter(
        chunk_size=config.chunk_size,
        chunk_overlap=config.chunk_overlap,
        separators=list(config.recursive_separators),
    )


def num_tokens_from_string(text: str) -> int:
    """Oblicza długość tekstu w tokenach przy użyciu skonfigurowanego tokenizera."""
    return len(encoding.encode(text))


def print_chunks_page_content(page_content: Iterable) -> None:
    """Wyświetla podstawowe statystyki i treść każdego fragmentu."""
    chunks = list(page_content)
    print(f"Liczba fragmentów: {len(chunks)}")
    short_str=""
    long_str=""
    for index, chunk in enumerate(chunks, start=1):
        body = getattr(chunk, "page_content", str(chunk))
        summary = (
            f"\n\nFragment {index} liczba znaków: {len(body)} "
            f"liczba tokenów: {num_tokens_from_string(body)}\n"
        )
        short_str+=summary
        long_str+=summary + body
    print(short_str)
    print()
    print(long_str)


def print_text_chunks(chunks: Sequence[str]) -> None:
    """Wyświetla statystyki fragmentów dla surowych sekwencji tekstu."""
    print(f"Liczba fragmentów: {len(chunks)}")
    for index, chunk in enumerate(chunks, start=1):
        summary = (
            f"Fragment {index} liczba znaków: {len(chunk)} "
            f"liczba tokenów: {num_tokens_from_string(chunk)}"
        )
        print(summary)
    print()
    for index, chunk in enumerate(chunks, start=1):
        summary = (
            f"Fragment {index} liczba znaków: {len(chunk)} "
            f"liczba tokenów: {num_tokens_from_string(chunk)}"
        )
        print(summary)
        print(chunk)
        print()


def handle_pdf_locally(uploaded_file: BytesIO, clean: bool = False) -> str:
    """Wyodrębnia tekst z pliku PDF przy użyciu PyPDF2."""
    logger.info("Przetwarzanie dokumentu lokalnie")
    try:
        uploaded_file.seek(0)
        pdf_reader = PdfReader(uploaded_file)
        texts = [(page.extract_text() or "") for page in pdf_reader.pages]
        output = "\n".join(texts)
        return clean_text(output) if clean else output
    except Exception:
        logger.exception("Błąd podczas lokalnego przetwarzania dokumentu")
        raise


def handle_pdf_remotely(uploaded_file: BytesIO, clean: bool = False) -> str:
    """Wyodrębnia tekst z pliku PDF przy użyciu Azure Document Intelligence."""
    logger.info("Zdalne przetwarzanie dokumentu PDF")
    doc_intelligence_endpoint = os.getenv("AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT")
    doc_intelligence_key = os.getenv("AZURE_DOCUMENT_INTELLIGENCE_ADMIN_KEY")

    if not doc_intelligence_endpoint or not doc_intelligence_key:
        raise EnvironmentError("Brakuje konfiguracji Azure Document Intelligence.")

    document_intelligence_client = DocumentIntelligenceClient(
        endpoint=doc_intelligence_endpoint,
        credential=AzureKeyCredential(doc_intelligence_key),
    )
    try:
        uploaded_file.seek(0)
        poller = document_intelligence_client.begin_analyze_document(
            "prebuilt-layout",
            body=uploaded_file,
            content_type="application/octet-stream",
            output_content_format="markdown",
        )
        result: AnalyzeResult = poller.result()
        return clean_text(result.content) if clean else result.content
    except Exception:
        logger.exception("Błąd podczas zdalnego przetwarzania dokumentu PDF")
        raise


def read_file_bin(file_name: str) -> BytesIO:
    """Odczytuje plik z dysku i zwraca jego zawartość binarną."""
    logger.info("Odczytywanie pliku %s", file_name)
    try:
        with open(file_name, "rb") as file:
            return BytesIO(file.read())
    except FileNotFoundError:
        logger.exception("Plik %s nie istnieje", file_name)
        raise


def save_file(file_name: str, data: str) -> None:
    """Zapisuje dane tekstowe na dysku z kodowaniem UTF-8."""
    logger.info("Zapisywanie pliku %s", file_name)
    with open(file_name, "w", encoding="utf-8") as file:
        file.write(data)


def clean_text(
    text: str,
    remove_comments: bool = False,
    put_html_tables_on_new_line: bool = True,
    preserve_linebreaks: bool = False,
    ) -> str:
    """Usuwa zbędne białe znaki i opcjonalnie usuwa komentarze HTML."""
    logger.info("Czyszczenie tekstu")
    text = re.sub(
        '(?<=<table>)(.*?)(?=</table>)',
        lambda match: match.group(0).replace('\n', ' '),
        text,
        flags=re.DOTALL,
    )
    patterns = {
        '\n+': '\n',
        ' +': ' ',
        r'\s<': '<',
        r'>\s': '>',
        r'\s\.': '.',
        r'\s,': ',',
        r'\s!': '!',
        r'\s\?': '?',
        r'\s:': ':',
        r'\s;': ';',
        r'\s\)': ')',
        r'\(\s': '(',
        r'\[\s': '[',
        r'\s\]': ']',
        r'\s\}': '}',
        r'\}\s': '}',
    }
    for pattern, replacement in patterns.items():
        text = re.sub(pattern, replacement, text)
    if put_html_tables_on_new_line:
        text = text.replace('<table>', '\n<table>')
    if preserve_linebreaks:
        text = text.replace('\n', '\n\n')
    if remove_comments:
        text = re.sub(r'<!--(.*?)-->', '', text, flags=re.DOTALL)
    return text

# Generowanie Markdown
Poniższa sekcja pokazuje wynik działania Azure Document Intelligence.

In [ ]:
file = read_file_bin(FILE_NAME)
md_file = handle_pdf_remotely(BytesIO(file.getvalue()))


# Segmentacja według nagłówków Markdown
Zacznij od segmentacji markdowna z Azure DI przy użyciu `MarkdownHeaderTextSplitter`. Zachowuje to hierarchię dokumentu, nadając każdemu fragmentowi metadane strukturalne przydatne w wyszukiwaniu wspomaganym oraz analizie.
Więcej informacji o splitterze znajdziesz w artykule [Jak dzielić Markdown według nagłówków](https://python.langchain.com/docs/how_to/markdown_header_metadata_splitter/).

In [ ]:
print(md_file)

In [ ]:
md_text_splitter = build_markdown_header_splitter(CONFIG)
md_header_splits = md_text_splitter.split_text(md_file)

print(f"Długość podziałów: {len(md_header_splits)}")

# Sprawdzanie fragmentów opartych na nagłówkach
Przejrzyj listę fragmentów, aby potwierdzić, że grupowanie nagłówków odpowiada logicznym sekcjom źródłowego PDF. Wczesna kontrola zmniejsza potrzebę późniejszego poprawiania wyników.

In [ ]:
print_chunks_page_content(md_header_splits)

# Analiza metadanych fragmentów
Każdy obiekt `Document` wygenerowany przez LangChain przechowuje zarówno fragment tekstu, jak i jego metadane (pochodzenie nagłówków, ścieżkę źródłową itp.). Analiza widoku serializowanego pokazuje, które atrybuty można ujawnić w indeksach wyszukiwania lub pulpitach analitycznych.

In [ ]:
print(md_header_splits[2].model_dump_json())


# Dopasowanie szczegółowości
Jeśli fragmenty na poziomie nagłówków są nadal zbyt duże dla docelowego okna kontekstu, dodaj kolejny splitter, aby uzyskać bardziej równomierne fragmenty.

## Dzielenie rekurencyjną strategią znakową
`RecursiveCharacterTextSplitter` przechodzi przez listę separatorów w ustalonej kolejności, aby w miarę możliwości zachować całe zdania. Dzięki temu powstają zwarte, spójne semantycznie fragmenty mieszczące się w docelowym limicie tokenów.
Dokumentacja: https://python.langchain.com/docs/how_to/recursive_text_splitter/.

In [ ]:
rct_text_splitter = build_recursive_splitter(CONFIG)

splits = rct_text_splitter.split_documents(md_header_splits)

print_chunks_page_content(splits)

## Normalizacja treści przed dzieleniem
Wykonaj lekkie czyszczenie, aby usunąć zbędne białe znaki, szum HTML i komentarze. Czystsze wejście daje bardziej przewidywalne granice fragmentów i ogranicza marnowanie tokenów.

In [ ]:
md_file_clean = clean_text(md_file,remove_comments=True)
print(md_file_clean)
save_file(FILE_NAME.replace(".pdf","-clean.md"),md_file_clean)

In [ ]:
md_header_splits_clean = md_text_splitter.split_text(md_file_clean)
print("Długość podziałów: " + str(len(md_header_splits_clean)))

In [ ]:
print_chunks_page_content(md_header_splits_clean)

In [ ]:
rct_text_splitter = build_recursive_splitter(CONFIG)

splits = rct_text_splitter.split_documents(md_header_splits_clean)

print_chunks_page_content(splits)

# Porównanie alternatywnych metod dzielenia
Różne zadania korzystają z różnych heurystyk dzielenia tekstu. Poniższe sekcje pokazują, jak biblioteki zewnętrzne mogą uzupełniać natywne splittery LangChain.

## Biblioteka Unstructured
`unstructured` oferuje dzielenie uwzględniające układ dokumentu, dostosowane do raportów i prezentacji. Jej splitter oparty na tytułach utrzymuje powiązane akapity razem, jednocześnie respektując konfigurowalne limity długości.

In [ ]:
NEW_AFTER_N_CHARS = CONFIG.chunk_size
MAX_CHARACTERS = CONFIG.chunk_size
COMBINE_UNDER_N_CHARS = CONFIG.chunk_overlap

elements = partition_md(text=md_file_clean)
print(f"Liczba elementów: {len(elements)}")

chunks = chunk_by_title(
    elements,
    multipage_sections=True,
    max_characters=MAX_CHARACTERS,
    new_after_n_chars=NEW_AFTER_N_CHARS,
    combine_text_under_n_chars=COMBINE_UNDER_N_CHARS,
    )

print(f"Liczba fragmentów: {len(chunks)}")
for index, chunk in enumerate(chunks, start=1):
    chunk_text = chunk.metadata.text_as_html if chunk.category == "Table" else chunk.text or ""
    summary = (
        f"Fragment {index} ({chunk.category}) długość={len(chunk_text)} "
        f"liczba_tokenów={num_tokens_from_string(chunk_text)}"
    )
    print(summary)

## Semchunk
`semchunk` stosuje segmentację wspieraną modelem, aby zachować ciągłość semantyczną. Skonfiguruj go docelowym modelem i rozmiarem fragmentu, aby prototypować granice chunków sterowane przez LLM.

In [ ]:
import semchunk 

semchunk_chunker = semchunk.chunkerify(CONFIG.semchunk_model, CONFIG.chunk_size)
semchunk_chunks = semchunk_chunker(md_file_clean)

print_text_chunks(semchunk_chunks)

# Następne kroki
- Sprawdź, czy poświadczenia Azure są skonfigurowane przed uruchomieniem zdalnego parsera.
- Poeksperymentuj z innymi ustawieniami `chunk_size` i `chunk_overlap`, aby dopasować je do okna kontekstu modelu embeddingowego.
- Zapisz przykładowe wyniki (`.md`, `.txt` i JSON z fragmentami), aby porównywać jakość wyszukiwania dla różnych strategii dzielenia tekstu.